In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece gradio pandas

In [ ]:
import re
import json
import pandas as pd
import torch
import gradio as gr
from threading import Thread
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer, BitsAndBytesConfig

In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
# MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

print("Loading tokenizer and model for:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    quantization_config=quant_config,
    trust_remote_code=True
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def extract_json_array(text: str):
    text = text.strip()
    text = re.sub(r"```json\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"```", "", text)

    start_index = text.find('[')
    end_index = text.rfind(']')

    if start_index != -1 and end_index != -1:
        text = text[start_index:end_index+1]

    try:
        data = json.loads(text)
        if isinstance(data, list):
            return data
    except Exception as e:
        print(f"JSON Parsing Error: {e}")
        try:
            text = re.sub(r',\s*\]', ']', text)
            return json.loads(text)
        except:
            pass

    raise ValueError("Model output is not a valid JSON array. Try increasing max_new_tokens or simplifying the request.")

def generate_dataset_stream(topic, record_count, style):
    schema = """
    [
      {
        "customer_id": "string",
        "age": "integer",
        "city": "string",
        "gender": "string",
        "subscription_plan": "string",
        "monthly_spend": "number",
        "purchase_category": "string",
        "last_login_days_ago": "integer",
        "churn_risk": "string",
        "customer_sentiment": "string"
      }
    ]
    """

    prompt = f"<|system|>\nYou are a synthetic data generator. Output ONLY a JSON array. <|user|>\nGenerate {record_count} realistic records about {topic} ({style}).\nSchema: {schema}\n<|assistant|>\n["

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    generation_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=1500,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    partial_text = "["
    for token in streamer:
        partial_text += token
        yield partial_text

    final_text = partial_text.strip()
    data = extract_json_array(final_text)
    df = pd.DataFrame(data)
    yield df

In [ ]:
with gr.Blocks(title="Synthetic Data Generator") as demo:
    gr.Markdown("# Synthetic Data Generator with LLM")

    with gr.Row():
        topic = gr.Textbox(value="ecommerce", label="Topic")
        n = gr.Slider(5, 100, value=20, step=1, label="Number of rows")
        style = gr.Textbox(value="online shopping customers", label="Style")

    btn = gr.Button("Generate")

    out_text = gr.Textbox(label="Streaming output", lines=12)
    out_df = gr.DataFrame(label="Generated data")

    def run_generation(topic, n, style):
        full_stream = ""
        for chunk in generate_dataset_stream(topic, int(n), style):
            if isinstance(chunk, str):
                full_stream = chunk
                yield full_stream, None
            else:
                yield full_stream, chunk

    btn.click(
        fn=run_generation,
        inputs=[topic, n, style],
        outputs=[out_text, out_df]
    )

demo.launch(debug=True, share=True)